# SPX 20/50 SMA regime backtest — token-free (Colab)

**Question (Jake's, refined):** it's not the *cross* that matters — it's the **character change**. For
months price sat ABOVE the 20 and the 20 sat ABOVE the 50 (healthy stack). Now price is spending real time
BELOW the 20 while the 20 bends down toward the 50. Does that combo actually precede weakness, or is it noise
a trending market shrugs off?

This tests **three** signals against the base rate so we know if any of them beat a coin flip:
- **A — classic cross:** the 20 crosses below the 50 (the thing everyone watches).
- **B — Jake's character-change:** 20 still ABOVE 50, but price spent most of the last 15 days BELOW the 20,
  the 20 is FALLING, and the gap to the 50 is CLOSING. (Fires *before* a cross — the early tell.)
- **healthy:** price>20 and 20>50 (what the last several months looked like — the control).

Run top-to-bottom. No keys, no tokens. If yfinance is blocked it falls back to Stooq automatically.


In [ ]:
import sys, subprocess
def _pip(p): subprocess.run([sys.executable,'-m','pip','install','-q',p])
try:
    import pandas as pd, numpy as np
except Exception:
    _pip('pandas'); _pip('numpy'); import pandas as pd, numpy as np

def load_spx():
    # primary: yfinance
    try:
        import yfinance as yf
    except Exception:
        _pip('yfinance'); import yfinance as yf
    try:
        df = yf.download('^GSPC', start='1990-01-01', progress=False, auto_adjust=True)
        if len(df):
            s = df['Close']
            s = s.iloc[:,0] if hasattr(s,'columns') else s
            return pd.Series(np.asarray(s).ravel(), index=df.index, name='close').dropna()
    except Exception as e:
        print('yfinance failed -> Stooq fallback:', e)
    # fallback: Stooq CSV, no key
    import urllib.request, io
    raw = urllib.request.urlopen('https://stooq.com/q/d/l/?s=^spx&i=d', timeout=30).read().decode()
    df = pd.read_csv(io.StringIO(raw), parse_dates=['Date']).set_index('Date').sort_index()
    return df['Close'].rename('close').dropna()

close = load_spx()
print('Loaded', len(close), 'bars:', close.index[0].date(), '->', close.index[-1].date(), '| last', round(float(close.iloc[-1]),1))


## Indicators + forward returns


In [ ]:
d = pd.DataFrame({'close': close})
d['sma20'] = d['close'].rolling(20).mean()
d['sma50'] = d['close'].rolling(50).mean()
d['above20']   = d['close'] > d['sma20']
d['s20_gt_s50'] = d['sma20'] > d['sma50']
d['s20_slope5'] = d['sma20'].diff(5)              # 20 rising(+) or falling(-)
d['gap']        = d['sma20'] - d['sma50']
d['gap_chg5']   = d['gap'].diff(5)                # gap widening(+) or closing(-)
d['frac_below20_15d'] = (~d['above20']).rolling(15).mean()  # share of last 15 days below the 20
d = d.dropna().copy()

# forward % return over next h trading days
for h in (5,20,60):
    d[f'fwd{h}'] = d['close'].shift(-h)/d['close'] - 1
# forward WORST drawdown over the next h days (min of next h closes vs today)
def forward_min(s,h): return s.shift(-1).rolling(h, min_periods=1).min().shift(-(h-1))
for h in (20,60):
    d[f'mdd{h}'] = forward_min(d['close'],h)/d['close'] - 1


## Signals


In [ ]:
# A) classic: 20 crosses below 50 (first day it happens)
d['sigA'] = d['s20_gt_s50'].shift(1).fillna(False) & ~d['s20_gt_s50']

# B) Jake's character-change: 20 STILL above 50, but price mostly below 20,
#    the 20 is FALLING, and the gap is CLOSING (20 bending toward 50). First day it triggers.
condB = (d['frac_below20_15d'] >= 0.5) & (d['s20_slope5'] < 0) & (d['gap'] > 0) & (d['gap_chg5'] < 0)
d['sigB'] = condB & ~condB.shift(1).fillna(False)

# control: healthy stack
d['healthy'] = d['above20'] & d['s20_gt_s50']


## Results — forward returns by signal vs base rate


In [ ]:
def summ(mask,label):
    sub = d[mask]; row = {'signal':label,'n':int(len(sub))}
    for h in (5,20,60):
        r = sub[f'fwd{h}'].dropna()
        row[f'{h}d avg'] = f'{100*r.mean():+.2f}%' if len(r) else '-'
        row[f'{h}d win'] = f'{100*(r>0).mean():.0f}%'  if len(r) else '-'
    m = sub['mdd60'].dropna()
    row['60d worst avg'] = f'{100*m.mean():.2f}%' if len(m) else '-'
    return row

rows = [summ(pd.Series(True,index=d.index),'ALL DAYS (base rate)'),
        summ(d['sigA'],'A: 20 crosses below 50'),
        summ(d['sigB'],'B: Jake character-change'),
        summ(d['healthy'],'healthy (px>20>50)')]
res = pd.DataFrame(rows).set_index('signal')
print(f'\n===== SPX FORWARD RETURNS BY SIGNAL  ({d.index[0].year}-{d.index[-1].year}) =====')
print(res.to_string())
print('\nRead it: does a signal row BEAT the base-rate row on avg return / win-rate / worst-drawdown?')
print('If it looks like the base rate, the signal times nothing.')


## Where are we RIGHT NOW + signal history


In [ ]:
last = d.iloc[-1]
print('===== CURRENT STATE', d.index[-1].date(), '=====')
print(f"close {last['close']:.0f} | sma20 {last['sma20']:.0f} | sma50 {last['sma50']:.0f}")
print(f"price {'BELOW' if not last['above20'] else 'above'} the 20   |   20 {'above' if last['s20_gt_s50'] else 'BELOW'} the 50")
print(f"20 slope(5d) {last['s20_slope5']:+.1f}  |  gap 20-50 {last['gap']:+.0f}  |  gap chg(5d) {last['gap_chg5']:+.1f}")
print(f"share of last 15 days spent below the 20: {100*last['frac_below20_15d']:.0f}%")
print('Jake B-signal ACTIVE right now:', bool(condB.iloc[-1]))

print('\nLast 12 B-signal dates (eyeball vs known events):')
for dt in d.index[d['sigB']][-12:]:
    f20 = d.loc[dt,'fwd20']; f60 = d.loc[dt,'fwd60']
    print('  ', dt.date(),
          f"fwd20 {100*f20:+5.1f}%" if pd.notna(f20) else 'fwd20  n/a ',
          f"fwd60 {100*f60:+5.1f}%" if pd.notna(f60) else '')


### How to read this / retune
- The point of the **base-rate row** is the honesty check: SPX drifts up over any random 20/60-day window, so a
  signal only *means* something if its forward return is materially WORSE (or its worst-drawdown deeper, or its
  win-rate lower) than that base row. Beating the base rate = edge. Matching it = the signal is theater.
- **Signal B is the one that maps to your chart** — it's designed to fire on the character change (price under
  the 20, 20 rolling toward the 50) *before* the cross. Watch its `n`: if it's rare, small-sample caveat applies.
- Retune B in the Signals cell: `frac_below20_15d >= 0.5` (how much time under the 20), `s20_slope5 < 0` (how
  hard the 20 must be falling), `gap_chg5 < 0` (gap must be closing). Loosen/tighten and rerun.
- Paste the two printed tables back to me and I'll read the verdict with you.
